# 042 NWCK Distance in the tree
## グラフにおける木の定義
- グラフが閉路を持たず、連結である。
- 連結なグラフの位数（点数）が n のとき、辺数が n-1 となる。
- 閉路を持たないグラフに隣接していない2点の間にどのように辺を加えても、閉路が1つだけできる。
- グラフの辺を1本でも消すと連結ではなくなる。
- グラフのどの2点間の道は1通りしかない（たどり方は必ず1通り）

- 木の任意の点から任意の点までの道筋は1通りであるため、その道筋の辺の長さを計算すれば、木の2点間の距離が求められる

## Newick format について
- 木をテキストで表記する手法の一種,系統樹などに用いられる
- 終わりは必ず; で示す
- 二分グラフよりも効率が良い。なぜなら、中間のノードを圧縮して表記するから
- 同字中間ノードに接続する葉ノードをneighbers という
```
    R
   / \
  U   D
 / \
A   B
```
- 葉ノードA と B は　内部ノードUのneighbors である
- 従って、((A, B)U, D)R; とくくることができる
- 中間ノードに名前がない場合(Uに名前がついていない場合)は省略可 (A, B) と書く
- 無根系統樹の場合、どの内部ノードを根とするかは自由に決められる。
- 根によってNewkick format は異なる

## Given
- 200個以下のノードで構成された40個以下のツリーが与えられる。
- 各ツリーについて、ツリー内の点 X, Y が与えられる。

## Return
- 一つの辺の長さを1 として、各ツリーとそれに含まれるノードペア間の距離を求めよ

## 考え方
- 共通祖先までたどり着くと( (Aを子に含む枝), (Bを子に含む枝) ) となる
- それ以上外に階層が存在していても考える必要はない
- つまり、A - B 間に存在する () の数を数えればよい

((A, B), (C, D))


In [ ]:
## ワード1とワード2 に挟まれる , (, ) を見れば何個層を上がる, 下がるかを考えることができるのでは

import re

def calc_tree_dist(word1, word2, tree):
    # ワード1とワード2の位置を見つける
    word1_idx = tree.index(word1)
    word2_idx = tree.index(word2)

    # ワード1とワード2の間の文字列を取得
    between_str = tree[min(word1_idx, word2_idx):max(word1_idx, word2_idx)]

    symbols = ["(", ")", ","]
    # ワード1とワード2の間の文字列から、括弧とカンマだけを抽出
    between_brackets = "".join([c for c in between_str if c in symbols])    
    
    # クエリに関係ない閉じた subtree は削除できる
    # ( 任意の文字 ) というパターンでは、((,), (,)) こういうパターンで壊れる
    # だから、再帰的に、( 任意の数のカンマ ) というパターンを除いていく
    while re.search(r'\(,*\)', between_brackets):
        between_brackets = re.sub(r'\(,*\)', '', between_brackets)
    
    # ここで、,),),), (,(,(, というパターンのみになる (クエリに関係ないサブの枝はすべて消える)
    # あとは、(の数を数えればよい
    # パターンとしては, 根-葉, 葉-葉　の2パターンがある
    # 親-子, 子-親の 場合は +2 する必要ない 

    # 例えば、)で終わるパターンは、根-葉のパターンで, +2 する必要がない
    if between_brackets.count(')') != 0 and between_brackets.count(')') + between_brackets.count(',') == len(between_brackets) and between_brackets.endswith(')'):
        return between_brackets.count(')')
    return between_brackets.count('(') + between_brackets.count(')') + 2

def get_answer(input_txt):
    lines = input_txt.strip().split("\n")
    # 3行ごとに, tree, word1 word2, 空行 というパターンで入力が与えられる
    answers = ""
    
    for i in range(0, len(lines), 3):
        tree = lines[i]
        word1, word2 = lines[i+1].split()
        dist = calc_tree_dist(word1, word2, tree)
        answers += str(dist) + " "
    
    return answers

In [ ]:
input_txt = """((((((((((((Abantias_moschata,(Clemmys_eburnea,Halichoerus_aeruginosus)),Bronchocela_rubicola),((((Ethmostigmus_aristotelis,Odonthurus_blythi),Kinosternon_caeruleus),((((Eurynorhynchus_tadorna,Spermophilus_carbo),Liasis_leucomelas),Thecla_nipalensis),Saiga_papuana)),Pusa_zonata)),Archispirostreptus_terrestris),Rhombomys_indicus),Recurvirostra_enhydris),Argynnis_standingii),Glareola_gordoni),(((((Avicularia_querquedula,((Cottus_quinquestriatus,Elaphe_carbo),Cypselus_galeatus)),Lamprolepis_politus),Iguana_griseus),Uncia_taczanowskius),Leuciscus_venulosa)),Procellaria_nyroca),Phelsuma_uluguruensis),((((((((Acanthogonatus_sujfunensis,(((((Anthropoides_barbatus,Gonyosoma_aestivus),Mabuya_tigris),((((Buthus_boschas,Mogera_capra),(Lanius_riparia,Platalea_cavimanus)),Cuora_peregusna),((Paraphysa_hipposideros,Rangifer_cepediana),Tetrao_docilis))),Phormictopus_milii),(Crocodylus_hungaricus,Eulabeia_serpentina))),(Ceratophrys_regius,(Gypaetus_jaculus,Lobipes_peregrinus))),(Boa_sinensis,((Kassina_melanostictus,Rhynchaspis_homeana),Porzana_orientalis))),(Chlamydotis_lesueurii,(Hadrurus_plumipes,Lobipes_subrufa))),(Allactaga_graeca,Homopholis_cioides)),(Bradyporus_novaeangliae,(Chelus_daurica,Ciconia_hypoleucus))),((((((((((((Ahaetulla_turtor,((Aquila_melanostictus,(Chamaeleo_linaria,(Citellus_minor,Emberiza_chamaeleontinus))),Oxyura_tenuirostris)),Rhesus_bimaculata),Haplopelma_hosii),Calotes_minor),(((Dyscophus_cambridgei,Teratolepis_vitulina),Gekko_uluguruensis),Saga_maihensis)),Rhynchaspis_completus),Panthera_ichthyaetus),Parabuthus_cambridgei),Buteo_major),Onychodactylus_pusilla),(Cinclus_angulifer,Kaloula_caerulea)),((((Aix_odoratus,Phrynosoma_nebularia),Salvelinus_cambridgei),Cygnus_modestus),Corallus_mehelyi))),(((((Alopex_iguana,(Amphiuma_vastus,((Enhudra_tetrax,Gekko_albertisii),Plethodon_cornix))),Avicularia_rubida),Tryngites_cygnoides),(Chettussia_unicolor,Middendorffinaia_cianeus)),Elseya_cyanogenys)),Ahaetulla_aureola);
Crocodylus_hungaricus Paraphysa_hipposideros

(Varanus_subrubrum,Vanellus_striatus,Tupinambus_dione,Trapelus_cristatus,Tiliqua_salvator,Thamnophis_torquata,Tetraogallus_meles,Terpsihone_borealis,Sus_ampullatus,Sternotherus_chinensis,Squaterola_licin,Spizaetus_cavirostris,Spermophilus_eulophotes,Spalerosophis_durus,Sitta_savignii,Sitta_acanthinura,Seokia_glareola,Seokia_chrysargos,Scincus_ignicapillus,Scincus_completus,Scaphiophryne_shadini,Salmo_prominanus,Rhinolophus_enydris,Recurvirostra_cingulata,Python_physalus,Pusa_pulcher,Pterocles_amethistina,Prunella_tristis,Porzana_bonasus,Pica_ussuriensis,Phrynomerus_comicus,Phormictopus_medici,Philacte_boyciana,Phelsuma_mystaceus,Perdix_fimbriatus,Pelusios_exquisita,Pelomedusa_salamandra,Pareas_mugodjaricus,Osmoderma_cocincinus,Oligodon_personata,Ninox_terrestris,Motacilla_tarda,Mogera_flavigularis,Meles_sieboldii,Marmota_sibiricus,Lyrurus_alcinous,Lutra_pyromelana,Lagenorhynchus_mexicanum,Iguana_dendrophila,Hysterocrates_lutra,Hottentotta_collaris,Homalopsis_notaeus,Hemiscorpius_decor,Haplopelma_penelope,Haplopelma_ferrumequinum,Haplopelma_cristatella,Grammostola_coloratovillosum,Gongylophis_nigra,Fregilegus_climacophora,Eutamias_zonata,Eubalaena_czerskii,Equus_multifasciata,Ephibolus_flavolineata,Emberiza_galericulata,Egretta_belliana,Cygnopsis_monilis,Cyclagras_nupta,Cuculus_fragrans,Ctenosaura_geniculata,Crocodylus_deremensis,Cottus_solitaria,Corytophanes_ferrumequinum,Columba_marinus,Coleonyx_scutulata,Clemmys_nippon,Citellus_multifasciata,Chelodina_dexter,Ceratophrys_subminiatus,Ceratophrys_smaragdina,Capella_wumuzusume,Caiman_nupta,Buteo_subniger,Buteo_rhymnus,Butastur_cepediana,Bradypodion_plumifrons,Balaenoptera_martensi,Ardea_japonica,Antilope_schneideri,Amphiuma_grossmani,Ambystoma_vipio,Almo_eulophotes,Agama_fischeri,Aegialites_carnivorus,Aegialifes_tetrax,Actitis_glutinosus)Vormela_karelini;
Emberiza_galericulata Aegialites_carnivorus

(,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,Chrysopelea_nupta,,Casarca_dominicus,,,,,,,,,,,,,,);
Casarca_dominicus Chrysopelea_nupta

((((((Acanthoscurria_grupus,Archispirostreptus_hypoleucus),Elaphe_cygnoides),(((((((((((Allactaga_krueperi,(((Buthacus_interiorata,Capella_shadini),Phrynops_onocrotalus),Leiolepis_mnemosyne)),Pseudorca_lesueurii),((((((((Callipogon_maculata,(((Chrttusia_doctus,(Damon_manul,(Enhydra_labiatus,Eublepharis_stellatum))),Podoces_flavigularis),Haliaeetus_wumuzusume)),Megaptera_venulosa),(Liasis_scutulata,Passer_martensi)),Salamandra_heterolepidotus),(Chlamydotis_indica,Mochlus_vertebralis)),Cygnopsis_melleri),Tetraogallus_squamatus),Epicrates_siebenrocki)),Python_flavescens),Eublepharis_lividum),Citellus_squaterola),(Sturnus_monorhis,Tadorna_pygmeus)),Pelomedusa_peregrinus),(Motacilla_scincus,sibiricus_pugatshuki)),Sturnus_karelini),Sceloporus_aegagrus)),((Chrysopelea_cranwelli,(((((((Columba_comicus,Nyctixalus_serpentina),Ctenotus_ferruginea),Grampus_elegans),Tiliqua_licin),Grus_chrysargos),Picus_colchicus),Sceloporus_dulkeitiana)),(Corytophanes_helena,Haliaeetus_gregarius))),Grus_coelestinus),(((((((((((Accipiter_turtor,((Basiliscus_caeruleus,Parabuthus_naumanni),(Gongylophis_punctatus,Pseudemys_asperum))),Recurvirostra_stejnegeri),Hysterocrates_tataricus),(((Dendrelaphis_vereda,(Mesoplodon_felderi,Tylototriton_variabilis)),(Dipus_ruficollis,Latastia_regius)),((((Gambelia_griseus,Nemachilus_fulva),Nemachilus_calligaster),((Hyperoodon_vitulina,Vipera_homeana),Pelodytes_caudata)),Pyrrhocorax_barbatus))),Caiman_sebae),Fulica_elegans),((((Buthus_coelestis,Eirenis_angulifer),((Circus_leschenaultii,((Enhydra_proteus,Larus_emarginatus),Tetrao_mnemosyne)),Corvus_minutus)),Rosalia_leucopsis),Minipterus_nebularia)),Bombycilla_cherrug),Falco_cinaedus),(Dipus_middendorffi,Saxicola_dendrophila)),Phrynomerus_clypeata),((Allactaga_arseniavi,(Athene_alpina,Boiga_diffidens)),(Aphonopelma_sepsoides,Ethmostigmus_marinus)));
sibiricus_pugatshuki Mesoplodon_felderi

((Acanthoscurria_brandtii,(Otocoris_totanus,Podoces_irregularis)),Acipenser_parahybana,(((((((((((((((((((((((((((Agama_naumanni,Nucifraga_altaicus),(((Cardiocranius_carolinensis,(Eumeces_sarasinorum,Sorex_scutulata)),((Lampropeltis_regius,Scolopax_nupta),Madagascarophis_grupus)),Neophron_floridana)),Upupa_emilia),Lasiodora_aegyptia),(Arenaria_mehelyi,Paramesotriton_querquedula)),(Anolis_versicolor,Terpsihone_alterna)),(Thecla_sp,Tupinambus_meles)),(((((((Balaena_gallicus,((Grus_duplex,Mylopharyngodon_lavaretus),Tropidurus_stejnegeri)),Monticola_longicollis),((Chlidonias_versicolor,Ziphius_dulkeitiana),((Megophrys_totanus,Streptopelia_nupta),Terpsihone_trigonopodus))),Tiliqua_ulikovskii),(Bombus_riparia,Dendrelaphis_scincus)),Gypaetus_nyroca),Motacilla_rosea)),Riparia_shadini),((Dendrobates_subrubrum,(Gerrhosaurus_yeltoniensis,Panthera_belliana)),(Grampus_timidus,Net_garullus))),(Capreolus_undulata,Uromastyx_erythrogastra)),Meles_rusticolus),Hyla_boa),(((((Chrysemys_eburnea,Monodon_marinus),Leiurus_caniceps),Crotaphytus_caelebs),Mabuya_celer),Coregonus_tinnunculus)),Ingerophrynus_compactus),(Bubulcus_bobac,Leiopython_lobatus)),Tryngites_nebrius),Marmota_exquisita),Citharacanthus_flavigularis),(((Arenaria_anatina,Carabus_mitratus),Bos_cianeus),Scaphiopus_bobac)),(Chalcides_rostratus,Spalerosophis_scincus)),((Camptoloma_cyanus,Pedostibes_haliaetus),Tylototriton_flavirufa)),Erpeton_rupestris),(Atrophaneura_doctus,Chen_cinerea)),(((((Branta_doctus,(Platalea_rosmarus,Thecla_weliczkowskii)),Saga_leucogeranus),Paramesotriton_hungaricus),Mustela_hendricksoni),Rhinolophus_brachydactyla)),Caiman_breitensteini),(((Alaus_plumifrons,Felis_strepera),Eschrichtius_taezanowskyi),Desnana_flavirufa)));
Monodon_marinus Balaena_gallicus

((((((((Acanthoceros_dorsalis,(((Dasypeltis_cioides,Tiliqua_cygnus),Litoria_pusilla),Gypaetus_mehelyi)),Glareola_penelope),Columba_franckii),((((((Bradypodion_arseniavi,Leiurus_anser),Halichoerus_karelini),((Micropalama_leucogeranus,Pleurodeles_armata),Siniperca_coelestinus)),Phrynocephalus_albicilla),Nemachilus_erythropus),(Colaeus_pallasii,Rufibrenta_torquatus))),(Ardea_stellio,Aythia_belliana)),(Hadrurus_vertebralis,Vormela_pygmaeus)),Bombus_leucophtalmos),(Acanthoceros_lobatus,(Lepus_corsac,Paraphysa_nyroca)),((((((((((((((((((((Acheron_chrysargos,Bos_leuconotus),((((((Babycurus_taezanowskyi,Corallus_laticauda),Brachypelma_placidus),Streptopelia_martensi),Saiga_squamatus),(((Glareola_minor,Minipterus_hilarii),Podoces_horridum),Grampus_caudatus)),Phyllopneuste_politus)),Remiz_martensi),Physignathus_tarandus),Saga_ridibundus),(((Almo_borealis,Egretta_kazanakowi),Nerodia_australis),Python_belliana)),Nyctixalus_albertisii),((((((((Aquila_korschun,Ketupa_helvetica),Cervus_carnivorus),Sitta_occitanus),(Gyps_tridactylum,Riparia_leuconotus)),(Chelodina_armeniacus,Lagenorhynchus_leucophyllata)),Cuora_interpres),Archispirostreptus_citrsola),Equus_mitratus)),Rhodostethia_reticulatus),(Balaena_cingulata,Citharacanthus_plathyrhychos)),((Babycurus_sebae,Philomachus_bifasciatus),(Pagophila_sanguinolentus,Telescopus_semipalmatus))),Chlidonias_doctus),Sorex_peregrinus),Enhydris_holbrooki),(((((((((Babycurus_lobatus,Sorex_longicaudata),(Lyrurus_quinquetaeniata,(Otis_rufinus,Porphyrio_semipalmatus))),Rhamphiophis_ferox),(Dyscophus_crucigera,Parus_microlepis)),(Pratincola_platyrhinos,Vipera_weliczkowskii)),Vulpanser_cingulata),Rhinolophus_torquata),Lamprophis_lividum),Rissa_mitratus)),Nerodia_leiosoma),Leiopython_vegans),Dipus_variegatus),(Alectoris_noctua,Epipedobates_hilarii)),Androctonus_tenuirostris));
Rissa_mitratus Nyctixalus_albertisii

(Alectoris_alpestris,Alpes_livia,Apus_australis,Aythia_atrigularis,Bradypterus_maurus,Bronchocela_blythi,Butastur_constricticollis,Capreolus_similis,Casarca_grunniens,Chelodina_piscator,Chen_oedicnemus,Chlidonias_trigonopodus,Ciconia_mystacinus,Clemmys_himalayanus,Corvus_chuatsi,Corvus_rhymnus,Cynops_oxycephalum,Cyriopagopus_armata,Dipus_picta,Epicrates_tridactylum,Furcifer_pardus,Gambelia_hemilasius,Geochelone_sauromates,Grampus_arenarius,Gypaetus_eulophotes,Hadogenes_aegyptia,Hadogenes_multituberculatus,Hadrurus_chinensis,Haliaeetus_vitulina,Hirundo_vanellus,Homopholis_alpina,Ingerophrynus_galactonotus,Laudakia_alcinous,Laudakia_perrotetii,Leiocephalus_atrigularis,Machetes_doctus,Mareca_dominicus,Megophrys_daurica,Melanocoryhpa_collybitus,Mergus_cavirostris,Mesoplodon_erythronotus,Mochlus_edulis,Monticola_taxus,Moschus_subcinctus,Mylopharyngodon_coloratovillosum,Natriciteres_clinatus,Neolycaena_glottis,Odobenus_subrufa,Ortigometra_tentaculatum,Pachydactylus_placidus,Pagophila_dignus,Paraphysa_musculus,Parus_lividum,Perdix_doctus,Phrynosoma_vittatus,Phyllopneuste_aegyptia,Podoces_mystacinus,Poephagus_laevis,Psammophis_semipalmatus,Pterinochilus_capra,Pyrrhocorax_argali,Querquedula_flavirufa,Rhesus_himantopus,Saga_galactonotus,Salmo_corsac,Saxicola_nyroca,Scaphiophryne_semipalmatus,Scorpio_dignus,Squaterola_rudicolis,Sturnus_hongkongensis,Sus_flava,Tadorna_calligaster,Teratoscincus_leucoptera,Terpsihone_pygmeus,Thecla_dahurica,Thecla_porphyrio,Tryngites_boa,Tursiops_placidus,Ursus_licin,Vanellus_flammea,Vipera_taezanowskyi,Vulpanser_salvator,crecca_leucopsis)Acanthosaura_lavaretus;
Dipus_picta Neolycaena_glottis

(((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((())))))))))))))))))))))))))))))))))))Oligodon_dignus)))Mergus_pelagicus)))))))))))))))))))))))))))))))))))))))))))))))))))))))))));
Mergus_pelagicus Oligodon_dignus

(Actitis_cocincinus,(Aegialites_tetrax,(Arctomys_microlepis,Turdus_naumanni)),((((((((((((((Ambystoma_coturnix,(Ciconia_rapax,Cuculus_minor)),((((Anser_classicus,(((((Castor_percnopterus,Ctenotus_czerskii),Gypaetus_ridibundus),(Dendrobates_calyptratus,Lamprophis_nyroca)),Limnaeus_adspersus),Erpeton_Bernicla)),((Brachypelma_plathyrhychos,(Ophisops_collaris,Thymallus_ochropus)),(((((((Budytes_communis,Ophisops_fragrans),Cuon_tinctorius),(Erpeton_totanus,(Trachemys_comicus,Uroplatus_bobac))),Halichoerus_kopsteini),Dafila_australis),Cuora_enydris),Chrttusia_buccata))),Himantopus_porzana),(((Bombycilla_daurica,(((((((Capella_karelini,Uromastyx_macqueni),Chelodina_vermiculatus),Numenius_hemionus),Vulpes_fragrans),(Trachemys_rufodorsata,Varanus_stellio)),(Gerrhosaurus_licin,(((Minipterus_mirabilis,Pyxicephalus_tanganicus),Nemachilus_molurus),Polypedates_clinatus))),Minipterus_botnica)),(Pelodytes_totanus,Tadarida_nigropalmatus)),Porzana_major))),((((Branta_leucophyllata,(Dipus_glacialis,Hydrochelidon_buccata)),Dryobates_rufodorsata),((((Ctenosaura_kopsteini,Uncia_cepediana),Squaterola_madagascariensis),Mustela_flavescens),(Philacte_cepediana,Saxicola_aureola))),(Kassina_deserti,Pituophis_minor))),(Lyrurus_trianguligerus,Pachydactylus_albicilla)),Lystrophis_saxatilis),((Camptoloma_cyanea,Osmoderma_bairdii),(Pareas_javanica,Rangifer_godlewskii))),(Kinixys_canagica,(Otis_macularius,(Pogona_communis,Sericinus_porzana)))),(Enhydra_melanoleucus,Leiopython_crassicauda)),(Python_gregarius,sibiricus_crocodilus)),Sturnus_catenifer),Cuora_livia),Arctomys_maculatum),((Eirenis_classicus,Phasianus_longipennis),Geochelone_virgo)),(((Bombus_buccata,(((Eschrichtius_brongersmai,Haliaeetus_carnivorus),(((Hysterocrates_carnivorus,Myotis_not),Nhandu_japonensis),Melanocoryhpa_quadrivirgata)),Oenanthe_armata)),Larus_albopillosum),((((Cynops_squaterola,(Epipedobates_campestris,Grus_murinus)),Terpsihone_agama),Gecarcinus_clericalis),Mylopharyngodon_corsac))));
Ophisops_fragrans Tadarida_nigropalmatus

(Actitis_paganus,Alaus_leucotus,((((Alaus_piscator,Riparia_dubius),Porzana_cinaedus),Buthus_calamita),((((((((((((((((((((((((Allactaga_turtur,Asthenodipsas_stellatum),((Eutamias_albirostris,Hottentotta_galactonotus),Ruticilla_relictus)),Parabuthus_zagrosensis),(Ctenosaura_chamaeleontinus,(Ctenotus_zenobia,Cuculus_ferox))),(Alopex_hyperboreus,Pyxicephalus_riparia)),Branta_flavirufa),Rangifer_docilis),(Euspiza_dives,Neolycaena_haliaetus)),Macrorhamphus_sarasinorum),((((((((((Anas_turneri,Mogera_obsoleta),Corvus_spilota),Myotis_Bernicla),Gazella_dorsalis),Pyrgilauda_conicus),(Lanius_taxispilota,Streptopelia_latiscutatus)),Nemachilus_boschas),(Aquila_macqueni,Geochelone_carbonaria)),(Desnana_castus,Kinixys_scabra)),Osmoderma_ussuriensis)),(Canis_leucoptera,(Meles_maihensis,(Parabuthus_torquata,Ptychozoon_blakistoni)))),Bradypodion_geyri),((Limnaeus_tricolor,sibiricus_epops),Spermophilus_jubata)),Sturnus_middendorffi),Vulpanser_himantopus),((Anodonta_cinclus,Burhinus_pulchra),Phormictopus_tetrax)),Grampus_cinereus),(Bradypodion_marmorata,Notophthalmus_azureus)),Chrysopelea_albocinctus),((Nyctixalus_carolinensis,Pelodytes_cianeus),Scorpio_stylifer)),(((((Ameiva_melleri,Falcipennis_xanthocheilus),Tetrao_rapax),(Cardiocranius_schokari,Phylloscopus_cavirostris)),Scolopendra_grunniens),(((((((((((Anolis_mlokosiewiczi,(Myotis_chukar,((Parabuthus_cliffordii,Tursiops_pulchra),Phelsuma_teguixin))),(Oceanodroma_bedriagai,Ursus_lepturus)),Monticola_krueperi),(((((Athene_dignus,Chalcides_interpres),Balaena_smithi),(((Clemmys_leucocephala,(Gallinago_crispus,Iguana_arenarius)),Nemachilus_rubicola),Lamprophis_cristatellus)),Sterna_scalaris),Milvus_emarginatus)),((Anthropoidae_proteus,Ctenotus_imperator),Cuora_filipjevi)),Heteroscodra_baeri),Pusa_hirundo),Furcifer_ocellatus),(((Cyriopagopus_leschenaultii,Hysterocrates_aristotelis),((Dafila_argali,Eublepharis_viridescens),Damon_hyperboreus)),Gyps_glutinosus)),Brachypelma_borealis),Thamnophis_sieboldii))),((Cottus_conicus,Cuculus_oedicnemus),Synthliboramphus_manul)),Micropalama_xanthocheilus),(Buthacus_dorsalis,Desnana_melanuroides))));
Branta_flavirufa Limnaeus_tricolor

((((((((Abantias_cliffordii,(Grampus_compactus,Porphyrio_holbrooki)),Eurynorhynchus_fasciata),(((Alopex_tanganicus,Cuon_personata),Bradypodion_sphenocercus),(Eremophila_guttata,Monachus_pugnax))),Nhandu_alpinus),((((Alpes_laticauda,(((((Antilope_proteus,(Athene_pulchripes,((Chrysopelea_exanthematicus,Lyrurus_sauromates),Margaritifera_caesius))),Mustela_americanus),(Cinclus_peregusna,Syrrhaptes_versicolor)),Latastia_emarginatus),Hysterocrates_fragrans)),Pyxicephalus_clinatus),((((Damon_avinivi,(Iguana_pelagicus,Lanius_daurica)),Dyscophus_insularis),Pusa_tetrax),Heterodon_japonica)),(((((((((((Alpes_oenanthe,(Limnaeus_rufina,Rhombomys_tarandus)),Machetes_cyanea),Lycodon_uncia),(Cynops_yeltoniensis,Eudrornias_anser)),Phalaropus_paradoxus),(((((((Capreolus_walti,Eschrichtius_plumipes),(((Monodon_peregrinus,Ziphius_physalus),Xenochrophis_grunniens),Phalacrocorax_fallax)),Uromastyx_rupestris),(((Grammostola_falcinellus,Gypaetus_personata),((Pagophila_acutus,Phelsuma_truncatus),Zosterops_nebrius)),Meles_chrysaetos)),Tetrao_cioides),((Dipus_turtor,Strepsilas_fulvus),Lepus_cinclus)),(Eudramias_carinatus,sibiricus_ignicapillus))),Diomedea_citrsola),(Coturnix_oedicnemus,(Gambelia_fasciolata,Iomachus_fernandi))),((Charadrius_soloensis,Platalea_atrigularis),(Cinclus_nigropalmatus,((Grus_smithii,Lepus_fragrans),(Pelusios_celeber,crecca_scincoides))))),(Arenaria_laticauda,(Pachydactylus_cherrug,Podoces_corticale))),Apodora_leschenaultii))),((Chen_porphyrio,Ketupa_jubata),Erpeton_parahybana)),(Allobates_calamita,((Chalcides_maurus,Dipus_sanguinolentus),Sus_gallicus))),Acanthoceros_equestris,(Alauda_oedicnemus,(((Holaspis_salamandra,(Opheodrys_mystaceus,Poephagus_maurus)),(Laudakia_fischeri,Mustela_dactylisonans)),(Rosalia_himalayanus,Selenocosmia_lepturus))));
Eurynorhynchus_fasciata Gambelia_fasciolata

((Acanthis_nigriceps,((((((((((((Acipenser_montela,((((Boiga_caudatus,(Cardiocranius_variabilis,Kaloula_pulcher)),Xenochrophis_casualis),Budytes_duplus),Hemitheconyx_caudatus)),((Babycurus_rufinus,Mesoplodon_compactus),Eudramias_stellatum)),((((Apodora_leucophyllata,Enhydra_murinus),(Nerodia_savignii,Saga_spinifera)),Pedostibes_marmoratus),Ptychozoon_teniotis)),Pyrgilauda_terrestris),Numenius_macularius),Riparia_filipjevi),Anthropoidae_taczanowskius),((((Alcedo_mlokosiewiczi,(Archispirostreptus_mlokosiewiczi,Crocodylus_paradisi)),(Homopholis_ceterus,(((Lycodon_galeatus,Tupinambus_fischeri),((Phylloscopus_deremensis,Sericinus_garullus),Vulpes_equestris)),Seokia_hemionus))),((Bos_nelsonii,Underwoodisaurus_striatus),Cyclemys_avocetta)),(((Archispirostreptus_grunniens,(Parus_quinquetaeniata,Regulus_calligaster)),((Boa_constricticollis,Neolycaena_gregarius),Tadarida_heterolepidotus)),((Argynnis_vertebralis,Vulpanser_stellio),Podoces_leptochelis)))),Limnaeus_unicolor),Nyctixalus_dahurica),(((((((((((Canis_microlepis,(Larus_docilis,Oceanodroma_spilota)),Leuciscus_relictus),Lepus_miliaris),Casarca_rubida),Grampus_weliczkowskii),Rangifer_citrsola),Clemmys_vereda),(Limosa_caudatus,Vulpes_milii)),Ophisops_crispus),(((Charadrius_nebularia,Pelusios_difficilis),Glareola_martensi),(Rissa_citreola,Tetrao_clinatus))),Pyxicephalus_leucotus)),Melanocorypha_calyptratus)),((((((((((Accipiter_morinellus,Charadrius_mackloti),Pica_labiatus),((((((((((Capeila_sibilans,Tiliqua_margaritifera),Procellaria_schokari),Middendorffinaia_novaeangliae),Chalcides_duplus),((Epicrates_leiosoma,Pterocles_cynodon),Platemys_taeniura)),Uroplatus_pusilla),Litoria_varius),Pusa_battersbyi),(((Dyscophus_ceterus,Scolopendra_hyperboreus),Ruticilla_versicolor),Underwoodisaurus_squamatus)),Corytophanes_ichthyaetus)),Citellus_barroni),((Corvus_leucorodia,Pterinochilus_septentrionalis),Pica_cenchria)),Chrttusia_leucorodia),(Milvus_hardwickii,Pelecanus_stagnalis)),Asthenodipsas_cyanogenys),(((Callipogon_gregarius,Enhydra_flammea),Carabus_clericalis),Oedura_arcticus)),Teratolepis_lehmanni),Acheron_mitratus);
Ophisops_crispus Vulpes_equestris

((((((Abantias_montela,Eublepharis_galeatus),Panthera_teniotis),Branta_sepsoides),((((((Alcedo_rubicola,Tropidurus_rosea),((Leuciscus_alpinus,Moschus_sauritus),Pagophila_valliceps)),(Dasypeltis_heterolepidotus,Underwoodisaurus_heudei)),Vipera_monedula),(((((((((((((((Alpes_tentaculatum,Circus_imperator),Dipus_caudata),Lepus_daurica),Heterodon_gordoni),(((Arenaria_alpestris,Mustela_brachydactyla),Synthliboramphus_nipalensis),(((((((Hemitheconyx_godlewskii,Macrorhamphus_leuconotus),(Litoria_carolinensis,(Motacilla_glareola,Oligodon_reinwardti))),Pseudorca_decorus),Tadorna_limosa),Perdix_tigrinus),Kinixys_castus),Phormictopus_cyanogenys))),(Ceratophrys_heudei,Pratincola_pugnax)),Haplopelma_guttata),(((((Antaresia_tenuirostris,(Argynnis_mystaceus,Mylopharyngodon_horridum)),Vulpes_ocellatus),Nyroca_castaneus),Liasis_stejnegeri),Chlidonias_dauricus)),(Pelecanus_thibetanus,Teratolepis_cioides)),((Chen_dorsalis,Chlamydosaurus_arenarius),Pterinochilus_paganus)),((Chlamydosaurus_pendulinus,Minipterus_saxatilis),Gambelia_hipposideros)),Moschus_fischeri),(((Cygnus_atriceps,Squaterola_giganteus),Parus_czerskii),Dryobates_celer)),(Bronchocela_armata,Ruticilla_capreolus)),(((Gongylophis_situla,Poephagus_guttata),Pituophis_aristotelis),Vulpes_boschas))),(((((((Almo_nigra,Rhesus_saiga),Pelodytes_acanthinura),Chamaeleo_calvus),(Buteo_cepediana,Perdix_vegans)),((Cardiocranius_timidus,Passer_apus),Eulabeia_iankowskii)),Procellaria_sirtalis),((((((Aythia_perrotetii,(Babycurus_gallinago,Eudrornias_kuhli)),(Cygnopsis_mitratus,Parnassius_coelestinus)),Sterna_kraepelini),(Ortigometra_versicolor,Ursus_collybitus)),(Crocodylus_ussuriensis,Synthliboramphus_cyanus)),(((((Bradypterus_maldivarum,(Sphenops_erythropus,Thamnophis_himalayanus)),Dipus_climacophora),Emydura_peregrinus),Plegadis_vitulina),Enhudra_rubicola))))),Circus_albocinctus),((Accipiter_eulophotes,Alectoris_hispida),Cypselus_leucogeranus),(Aix_marcianus,Oceanodroma_constricticollis));
Perdix_tigrinus Tadorna_limosa

(Uromastyx_baeri,Underwoodisaurus_hemilasius,Tylototriton_hassanica,Triturus_albirostris,Totanus_oenanthe,Spalerosophis_clinatus,Siniperca_cavirostris,Scincus_fluviatilis,Saxicola_hilarii,Saiga_daurica,Rhynchophis_stellio,Python_leucorodia,Physignathus_labiatus,Phoca_arcticus,Pelomedusa_dolosus,Passer_helvetica,Paraphysa_coloratovillosum,Paradoxornis_crocodilus,Pandinus_medirostris,Otocoris_gregarius,Nyctixalus_coelestis,Nucifraga_acuta,Neolycaena_piceus,Monodon_hosii,Mergus_avicularia,Limnodromus_rufodorsata,Liasis_gobio,Lepidobatrachus_pusilla,Larus_lagopus,Kassina_classicus,Hydrosaurus_ampullatus,Halichoerus_squamatus,Haliaetus_ciliatus,Haliaeetus_semipalmatus,Glareola_parahybana,Gambelia_microlepis,Euspiza_cynodon,Eubalaena_grandis,Ethmostigmus_tridactylum,Ethmostigmus_paganus,Epicrates_moschata,Egretta_torquatus,Dipus_cyanogenys,Diomedea_galeatus,Dahurinaia_merganser,Cyclagras_lineatus,Cyclagras_hyemalis,Ctenosaura_kazanakowi,Cottus_personata,Coleonyx_schrencki,Circaetus_lehmanni,Chrttusia_aegagrus,Chlamydotis_vereda,Charadrius_albertisii,Capra_occitanus,Capra_japonica,Capella_sibiricus,Candoia_garullus,Candoia_dolosus,Buteo_savignii,Buteo_mysticetus,Branta_linaria,Branta_albicilla,Bradypodion_tanganicus,Brachyramphus_medirostris,Boiga_metallica,Boa_fernandi,Boa_albertisii,Balaenoptera_smithii,Arenaria_acuta,Aphonopelma_scalaris,Apalone_meles,Antilope_ulikovskii,Antaresia_vereda,Anser_caryocatactes,Anolis_pyromelana,Alloporus_leiosoma,Alectoris_proteus,Alaus_relictus)Vipera_galericulata;
Charadrius_albertisii Boa_albertisii

(((Acanthoceros_chrysaetus,Cottus_tricolor),((Alectoris_chrysaetus,((((((Alopex_licin,Arctomys_piceus),Minipterus_pholeter),Dryobates_leucophtalmos),Thymallus_geniculata),Babycurus_mexicana),Petrocincla_macularius)),Platemys_boulengeri)),Acanthogonatus_albocinctus,((((((((((((Acanthosaura_major,((((((Ahaetulla_cepediana,Selenocosmia_musicus),Hemitheconyx_pusilla),(Boa_aegagrus,Mogera_rubida)),Lampropeltis_hemionus),(Platemys_alpina,Uncia_garullus)),((((Cuculus_hirundo,(Haliaetus_catenifer,Lanius_eulophotes)),Scincus_dauricus),Pelecanus_sphenocercus),(Eudramias_gallicus,Scaphiophryne_linaria)))),(Ambystoma_pyromelana,((Chelydra_vanellus,Lepidobatrachus_fulvus),(Dafila_azureus,Vormela_hasselquistii)))),(((((((Bufo_glareola,Mergus_atrigularis),(Phalaropus_cinereus,(Plegadis_exanthematicus,(Riparia_macularius,Ursus_salamandra)))),Scaphiopus_nigropalmatus),Kinixys_gratiosa),Ctenosaura_pelagicus),Tupinambus_gallicus),Pandinus_multituberculatus)),Saxicola_maculata),(Accipiter_carinatus,Capra_mexicana)),((((Alcedo_govinda,(Coturnix_pallasii,Liasis_cianeus)),((Colaeus_crassicauda,Vipera_barbatus),Coregonus_temminckii)),((((((((((((((Alloporus_macqueni,Net_physalus),Atrophaneura_isabellina),(Apalone_tentaculatum,Calotes_helvetica)),Glareola_glutinosus),Arenaria_bedriagai),(Bradypodion_vereda,Triturus_climacophora)),Arctomys_microlepis),(Fuligula_guttata,((Kaloula_fragrans,Pogona_kazanakowi),Lagenorhynchus_lopatini))),((Anas_gigas,Falco_cyanea),(Casarca_proteus,Pelusios_pyromelana))),((Mylopharyngodon_oedicnemus,Xenochrophis_apus),Sus_rufina)),Zosterops_spilota),Phasianus_clericalis),((Dryobates_argali,Eirenis_flavigularis),Neolycaena_communis)),Bradyporus_clypeatus)),Netta_microlepis)),(Columba_weliczkowskii,(Prunella_corone,Uroplatus_macrops))),Rhacodactylus_politus),Hydrochelidon_monachus),((Acanthoscurria_salamandra,Mareca_cyanus),(((Calotes_sieboldii,Mabuya_cherrug),Larus_saiga),Procellaria_stagnalis))),Gerrhosaurus_ocellatus),Otis_bicoloratum));
Otis_bicoloratum Pandinus_multituberculatus

((((((((((Aegypius_horridum,Chelodina_ochropus),Emydura_cinereus),((Enhudra_caucasicus,Scincus_rufina),(Notophthalmus_hodgsoni,Sphenurus_lavaretus))),(Dasypeltis_completus,Rosalia_gregarius)),((Grampus_verrucosus,Tylototriton_stejnegeri),((Pelecanus_elaphus,sibiricus_doriae),Uroplatus_carbo))),Pagophila_flavolineata),Eutamias_marinus),Meles_capreolus),(((Cyclemys_krueperi,Neophron_dahurica),Passer_ignicapillus),Vulpanser_macrops)),Anodonta_teniotis,(((((((((((((Anthropoidae_clypeata,Passer_clypeata),(Falcipennis_jaspidea,Ziphius_geniculata)),((((((Bombycilla_leucogaster,((Gekko_politus,Lutra_tuberculosus),Rhynchophis_turneri)),(Calidris_maihensis,((((((((Coturnix_shadini,Netta_lineatus),Melanocorypha_fuscatus),((Lobipes_rusticolus,Rhamphiophis_doctus),Nerodia_labiatus)),Ziphius_rapax),(Eulabeia_nigropalmatus,Phalaropus_boa)),Pelodytes_emarginatus),Elaphe_reticulatus),(Lepus_eremita,(Rufibrenta_australis,Syrrhaptes_scincoides))))),(Homopholis_resinifictrix,Nyctixalus_ochropus)),Oenanthe_dispar),(Hyperoodon_yeltoniensis,Osteopilus_hipposideros)),((Capeila_spilota,Vulpanser_casualis),(Passer_flavolineata,Recurvirostra_glacialis)))),(((Bronchocela_lavaretus,Remiz_guentheri),Fuligula_chukar),((Coenobita_apollo,Rhamphiophis_nebrius),Sphenurus_baibacina))),(((((((Archispirostreptus_alterna,(Capeila_medici,Myotis_hypomelus)),Pandion_pendulinus),Brachyramphus_picta),Hemitheconyx_clypeata),Cuculus_wislizeni),Turdus_gobio),Seokia_vitulina)),Nemachilus_medirostris),Motacilla_doctus),(Eulabeia_japonensis,Psammophis_helena)),Rufibrenta_flavescens),((Erpeton_mitratus,Squaterola_fernandi),(Scorpio_lobatus,Uncia_mongolica))),Pyrgilauda_melleri),Lystrophis_salamandra),((Antilope_trianguligerus,Phelsuma_hardwickii),Gekko_czerskii)));
Phalaropus_boa Scincus_rufina

((Acanthoceros_means,(Androctonus_meles,Pica_stellio)),(((Acanthoceros_perdix,Antaresia_albigula),((Aquila_canagica,Nucifraga_opimus),Euspiza_temminskii)),((Capeila_viridescens,Philacte_boulengeri),Mylopharyngodon_rufus)),(((((((((((((Actitis_stejnegeri,(Alcedo_tenuirostris,(((Babycurus_bonasus,(Rhombomys_radiata,Testudo_shadini)),Chrttusia_dahurica),Theloderma_cepediana))),Dyscophus_dexter),((Coregonus_graculus,(Gyps_quinquetaeniata,Monachus_prasina)),Ketupa_altaicus)),Aix_dentatus),((((Aythya_weberi,(((Balaena_nigropalmatus,Dafila_cyanus),Phoca_metallica),Chalcides_battersbyi)),Bronchocela_gebleri),((Cyclemys_tigris,Salvelinus_brevipes),Kaloula_licin)),((Ciconia_tigris,Tadarida_dexter),Sus_davidiana))),(((Chen_molurus,Tryngites_baeri),((((Chondropython_quadriocellata,Hadrurus_sirtalis),Glareola_timidus),Pseudorca_jubata),Totanus_flavirufa)),(Megaloperdix_longicollis,Oenanthe_japonensis))),Bradypterus_macularius),Osteopilus_lepturus),Synthliboramphus_undulata),((((((((((Bubulcus_kopsteini,Charadrius_hassanica),Telescopus_rubicola),Uncia_crassidens),Podoces_intermedia),(Pterocles_vermiculatus,Sitta_chuatsi)),(Lanius_glacialis,Siniperca_wogura)),((Bufo_orientalis,(Neolycaena_physalus,Scolopendra_dorsalis)),Saga_platyrhinos)),Cardiocranius_flavomaculatus),Eschrichtius_apollo),(Dryobates_carbonaria,(Gonocephalus_paradisi,Pica_javanica)))),Eulabeia_chrysaetus),((Asthenodipsas_temminckii,Gecarcinus_clypeatus),Mylopharyngodon_collybitus)),(((Alcedo_versicolor,(Branta_argentatus,Paramesotriton_tinctorius)),((Allobates_fischeri,((Anas_viscivorus,Gonocephalus_dentatus),Rhinolophus_hassanica)),(Gyps_atra,Net_uluguruensis))),((Apalone_meermani,(((Argynnis_kurilensis,((Leiocephalus_trigonopodus,Pareas_gallicus),Upupa_madagascariensis)),Coleonyx_cynodon),(Pusa_manul,Spizaetus_turtor))),((Chlamydosaurus_terrestris,Hottentotta_soloensis),((Eurynorhynchus_wogura,Oligodon_ruthveni),Lasiodora_dominus))))));
Glareola_timidus Megaloperdix_longicollis

((((((((((((((((((((Abantias_castaneus,Dendrobates_geniculata),(Boa_leucoryphus,Lycodon_acutus)),((Atrophaneura_stimsoni,Oceanodroma_holbrooki),Trachemys_squaterola)),(((((((Ahaetulla_horridum,(Dasypeltis_versicolor,Lycodon_tatarica)),Lystrophis_rosmarus),Leptobrachium_uncia),((Pandion_ion,Xenochrophis_scincoides),Procellaria_graeca)),Citharacanthus_miliaris),Pyxicephalus_mysticetus),Siniperca_rupestris)),(Rissa_himalayanus,Teratoscincus_boa)),(((Cyclemys_rufina,Mesoplodon_calidris),Rhodostethia_asperum),Ziphius_zenobia)),Scincus_dubia),Apalone_auriculatus),((Allactaga_hypomelus,Netta_thibetanus),Egretta_striatus)),(Boa_eremita,(Melanocoryhpa_hypomelus,Phormictopus_opimus))),Vanellus_cinereus),((((Colaeus_javanica,Eschrichtius_borealis),Phormictopus_verrucosus),Otis_miliaris),Petrocincla_guineti)),Micropalama_unicus),((Gambelia_vermiculatus,Sericinus_merganser),Pyxicephalus_modestus)),((((Babycurus_physalus,(Neophron_barroni,Vulpanser_vulpes)),(Natriciteres_dexter,Strepsilas_chrysaetos)),(Cervus_gratiosa,(Notophthalmus_franckii,Pterocles_maldivarum))),(Macrorhamphus_hassanica,Nyctixalus_unicus))),((Caiman_kraepelini,Python_situla),Pituophis_ridibundus)),((((Asthenodipsas_laevis,(Lampropeltis_leucostomum,Physignathus_seemani)),Notophthalmus_dolosus),Pterinochilus_piceus),(Camptoloma_tetrix,Odobenus_squamatus))),(Phyllopneuste_sudanensis,Salamandra_horridum)),Ceratophrys_aceras),(((((((((((Abantias_sauromates,((Eurynorhynchus_collectivus,Podoces_glottis),Lamprophis_jacksoni)),Thymallus_wumuzusume),((Ketupa_moschiferus,Thymallus_calidris),(Oxyura_fiber,Teratolepis_nasicus))),Rhesus_canorus),(Aix_calyptratus,Desnana_unicus)),Hadrurus_cepediana),(Equus_smithii,(Gekko_standingii,Otis_cristata))),((Actitis_bifasciatus,Laudakia_carinata),Perdix_wumuzusume)),Boiga_jaspidea),Platalea_clypeatus),(Certhia_hispida,Haplopelma_pusilla)),((((Acanthoscurria_radiata,(Pyrrhocorax_chukar,(Terpsihone_eximia,Tetraogallus_sibiricus))),(Apalone_emarginatus,Aphonopelma_hendricksoni)),Chondropython_platycephala),Aphonopelma_macularius));
Ziphius_zenobia Tetraogallus_sibiricus

(,,,,,,,,,,,,,,,,,,,,,,,,,Phasianus_iankowskii,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,Dyscophus_interpres,,,,,,,,,,,,,,,,,,,,,);
Dyscophus_interpres Phasianus_iankowskii

(Acanthis_mehelyi,Aix_plumifrons,((((((((((((((Anas_leucogaster,((((((Balaenoptera_caesius,Charadrius_novaeangliae),Opheodrys_timidus),Melanocorypha_solitaria),Tursiops_aristotelis),Phrynosoma_minor),Scolopendra_albocinctus)),((Furcifer_scincoides,Norops_variabilis),Ingerophrynus_walti)),Sericinus_decor),Phrynosoma_nigrolineatus),Budytes_caelebs),Sorex_leucomystax),((Bradyporus_corticale,(Cygnopsis_parahybana,Eryx_vereda)),Castor_caryocatactes)),Gonocephalus_vulgaris),(((Callipogon_radiata,Chalcides_alpinus),((Chelus_collybitus,Glareola_maculata),Remiz_rudicolis)),(((((Columba_caelebs,(((Grus_fasciolata,Tadarida_oxycephalum),Minipterus_apus),Mabuya_spaldingi)),((Dahurinaia_pholeter,Tryngites_heudei),Odobenus_spinifera)),Diomedea_catenifer),(Heteroscodra_shadini,Ovis_pulchra)),crecca_cianeus))),Cardiocranius_scincoides),(Bradypterus_gregarius,Cyriopagopus_ceterus)),(((Melanocoryhpa_chrysargos,Sericinus_longicaudata),Phalacrocorax_tarandus),(Philacte_cinclus,Sternotherus_alterna))),(Lasiodora_querquedula,Nhandu_serricollis)),(((((((((((((((Bombina_macrops,(Liasis_proteus,Monticola_amboinensis)),((Coleonyx_jaspidea,Passer_alpestris),Pituophis_battersbyi)),Damon_helena),Phelsuma_leucotus),Cervus_siebenrocki),(((((Candoia_krueperi,Cuculus_gregarius),(Cypselus_spilota,((Gyps_mexicanum,Prunella_perrotetii),Xenophrys_leschenaultii))),Chlidonias_altaicus),Vormela_hilarii),Numenius_margaritifera)),(Cygnopsis_macqueni,Lagenorhynchus_exquisita)),Branta_pygargus),Pyxicephalus_maculata),(Gongylophis_sibiricus,Lobipes_intermedia)),((Epipedobates_merganser,Turdus_yeltoniensis),Oedura_strepera)),Callipogon_tigrinus),(Dafila_nipalensis,(Leptobrachium_enydris,(Mustela_rudicolis,(Pelodytes_flava,Spalerosophis_paradisi))))),(Branta_melanuroides,Nerodia_glaucescens)),(((((Circaetus_dione,(Lycaenopsis_stellatum,Oxyura_leucorodia)),Spalerosophis_teguixin),Sericinus_erythronotus),Scolopax_novaeguineae),Heterodon_strepera))));
Odobenus_spinifera Pyxicephalus_maculata

(,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,Paraphysa_peregrinus,,,,,,,,,Rhombomys_longipennis,,,,,,,,,,,,,,,);
Paraphysa_peregrinus Rhombomys_longipennis

((Accipiter_standingii,(Anthropoides_gobio,((((Canis_pygmeus,((((((Egretta_thibetanus,Pachytriton_coturnix),Riparia_caudatus),sibiricus_cavimanus),(Emydura_buccata,(Gambelia_gecko,Netta_temminckii))),Tadarida_alpinus),((Gonyosoma_atrigularis,Philomachus_sarasinorum),Pyrgilauda_teniotis))),(Gyps_nigra,Theloderma_cioides)),Pyrrhocorax_fimbriatus),Tadarida_leucoryphus))),(((((((((Alpes_pallasii,(Egretta_kuhli,Haplopelma_dubius)),(Enhydra_punctatus,(Norops_platyrhinos,Uroplatus_pedo))),(Hemitheconyx_capra,Rhombomys_proteus)),(((Heteroscodra_gratiosa,(Mustela_monacha,(Nyctaalus_tridactylum,Tryngites_heterolepidotus))),(Kaloula_perdix,Rosalia_difficilis)),Varanus_marinus)),Chen_celer),Furcifer_cygnoides),(((Candoia_wumuzusume,Ethmostigmus_means),(Scorpio_nivicola,Sternotherus_durus)),(Damon_docilis,Sericinus_geyri))),(Chrysopelea_brandtii,(((Clemmys_guttifer,Remiz_aeruginosus),(((Cyclemys_fiber,Ninox_weberi),(Micropalama_fragrans,Porphyrio_colchicus)),Tryngites_bifasciatus)),Nyctaalus_standingii))),(((Aquila_saxatilis,Netta_sujfunensis),(Rhynchaspis_taxus,Tringa_subniger)),((Certhia_avinivi,(Chen_chinensis,(Dahurinaia_mitratus,Gecarcinus_irregularis))),Gongylophis_uluguruensis))),(((((((((((((Ameiva_glottis,(Dyscophus_armata,(Gavia_arcticus,Lepidobatrachus_hardwickii))),(Camptoloma_cyanogaster,Mabuya_vitulina)),Nhandu_squamatus),Phrynocephalus_erythronota),Pelodiscus_hyperboreus),((Limnodromus_fallax,Rhinolophus_scincoides),(Testudo_nigra,Uncia_scabra))),((Circaetus_erythronotus,Grammostola_coturnix),Totanus_subrubrum)),(Oenanthe_vereda,Pseudemys_fuellebornii)),(Citharacanthus_licin,(Egretta_zagrosensis,Hadogenes_mexicana))),((Spalerosophis_collaris,Xenopeltis_dubia),Uroplatus_canus)),Tursiops_varius),Siniperca_avinivi),(((((((Apodora_ibera,Chrysemys_leucophtalmos),Natriciteres_weliczkowskii),((Iomachus_sauritus,Lepidobatrachus_atrigularis),Ziphius_means)),Hemiscorpius_carolinensis),Lystrophis_pictus),((((Atrophaneura_dahurica,Sphenurus_kopsteini),(Dasypeltis_nebularia,Lyrurus_spilota)),Columba_alterna),Bombus_brevipes)),Leptopelis_rostratus)));
Nyctaalus_standingii Gyps_nigra

(,,,,,,,,,,,,Cervus_vulgaris,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,Phoca_physalus,,,,,,,,,,,,,,,,,,,,,,);
Cervus_vulgaris Phoca_physalus

(((((((((((((Abantias_bewickii,Equus_virgo),Ruticilla_acanthinura),((((Ardea_lineatus,Circus_citreola),Xenochrophis_subcinctus),((Bombyx_turtur,Vulpanser_sepsoides),Furcifer_imperator)),(Bombyx_monoceros,Ptyodactylus_dauricus))),((((((Anthropoidae_gallicus,(Bradyporus_hasselquistii,(Mochlus_indicus,(Rhodostethia_graeca,Upupa_galactonotus)))),Atrophaneura_flavescens),Ptyodactylus_carnifex),((Bradyporus_olivacea,Lobipes_dulkeitiana),Rhynchophis_scincus)),Odonthurus_microlepis),Corvus_leucoptera)),(Brachyramphus_salamandra,(Buteo_pachypus,(Citellus_prasina,Pyxicephalus_hassanica)))),Milvus_sebae),Corallus_aristotelis),((Anodonta_scincus,(Arctomys_citrsola,(((((((((((Arenaria_cyanus,Bradypterus_monachus),Bombus_jacksoni),(Lamprolepis_kraepelini,Nemorhaedus_guttifer)),Minipterus_dominus),Ovis_lobatus),((Equus_turtur,Uromastyx_macularius),Nemachilus_bicinctores)),(((((((Burhinus_geniculata,Caiman_coturnix),Margaritifera_elaphus),Coleonyx_anachoreta),Dendrelaphis_purpurascens),((Coenobita_albirostris,((Phalaropus_pulchra,Varanus_squaterola),Trionyx_cyanochloris)),Rangifer_dubia)),Chamaeleo_arseniavi),Circaetus_trigonopodus)),Netta_nipalensis),(Eudrornias_nasicus,Uromastyx_carbo)),Leptobrachium_carolinensis),Telescopus_canus))),((((Anthropoidae_colubrinus,(((((((Athene_cygnoides,(((((Caiman_aegagrus,Gazella_meles),Fregilegus_aureola),Natriciteres_cioides),Desnana_semipalmatus),Capra_ammon)),Melanocoryhpa_plathyrhychos),Falcipennis_means),Salamandra_meermani),Netta_musculus),Parnassius_bobac),Bubulcus_taxus)),Chettussia_cancerides),Himantopus_capra),Cygnus_alba))),Chondropython_eburnea),Fuligula_durus),Plegadis_himalayanus),(Bradypodion_haliaetus,Epicrates_graculus)),(Aegialifes_bairdii,((((Balaena_kurilensis,(Cuculus_livia,Sceloporus_paganus)),(Liasis_corticale,Psammophis_alterna)),((((((Budytes_hispida,(Iguana_picta,Myotis_tinctorius)),Regulus_picta),(Candoia_decor,Rhesus_orientalis)),Iomachus_hilarii),Regulus_manul),(Natriciteres_dexter,Nyctixalus_emarginatus))),(Lamprophis_angulifer,Neolycaena_lagopus))),Almo_fuellebornii);
Rangifer_dubia Pyxicephalus_hassanica

(Alcedo_laevis,Alloporus_uncia,Ambystoma_sanguinolentus,Amphiuma_grunniens,Anthropoidae_heudei,Anthropoides_epops,Apalone_nigropalmatus,Ardea_serpentina,Arenaria_pygargus,Atrophaneura_brachydactyla,Balaena_subruficollis,Boa_cavimanus,Branta_orientalis,Canis_eremita,Canis_lepturus,Ceratophrys_altaicus,Cervus_indicus,Chlamydotis_ferruginea,Chlamydotis_japonica,Chondropython_exquisita,Chondropython_seemani,Colaeus_crassicauda,Corvus_hypomelus,Ctenosaura_hasselquistii,Ctenotus_trigonopodus,Cuculus_albopillosum,Cygnopsis_calvus,Cygnus_grupus,Cyriopagopus_torquata,Dendrelaphis_atrigularis,Dryobates_boschas,Eublepharis_fernandi,Eulabeia_buccata,Gavia_oenanthe,Gazella_aspera,Grampus_papuana,Grampus_walti,Gyps_crocodilus,Hadrurus_floridana,Halichoerus_cornix,Himantopus_amboinensis,Hysterocrates_dubius,Hysterocrates_mycterizans,Kinosternon_stejnegeri,Lagenorhynchus_acuta,Lampropeltis_fasciata,Larus_africanus,Lepidobatrachus_bairdi,Leuciscus_indica,Lycaenopsis_niloticus,Megaptera_krueperi,Morelia_turtur,Mylopharyngodon_canagica,Neophron_nigrolineatus,Netta_laevis,Oenanthe_doctus,Oenanthe_kraepelini,Paramesotriton_maldivarum,Parus_cliffordii,Pelecanus_limosa,Pelodiscus_marcianus,Phalacrocorax_circia,Philacte_longicollis,Phrynomerus_bobac,Phyllopneuste_cenchria,Platalea_auratus,Plethodon_bairdi,Poephagus_gecko,Porzana_dignus,Porzana_rufus,Psalmopoeus_dubius,Ptychozoon_celeber,Ptyodactylus_grupus,Python_rufinus,Pyxicephalus_hyperboreus,Scolopax_erythronota,Sericinus_zenobia,Tadorna_fuscatus,Teratolepis_tinctorius,Tetraogallus_subrufa,Tetraogallus_taezanowskyi,Thamnophis_heliaca,Turdus_atra,Turdus_weliczkowskii,Uncia_cinerea,Ursus_tarandus,Xenopeltis_fallax,Xenophrys_constrictor,Xenophrys_dignus,Xenophrys_hispida)Abantias_variegatus;
Dendrelaphis_atrigularis Ceratophrys_altaicus

(((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((())))Telescopus_uncia))))))))))))))))))))Pachytriton_gobio))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))))));
Telescopus_uncia Pachytriton_gobio

((Acanthoceros_ferruginea,(((((((((((Alopex_elaphus,((((((((((Balaenoptera_daurica,Vulpes_eulophotes),(Cuculus_flavirufa,Salamandra_rubida)),Capella_deserti),Strepsilas_multifasciata),Budytes_fiber),Telescopus_guttata),Monachus_adspersus),((((Colaeus_temminckii,Megaloperdix_borealis),Strepsilas_riparia),(Leiopython_anachoreta,Scincus_porzana)),Hottentotta_aceras)),(Ephibolus_undulata,Onychodactylus_nebularia)),Columba_carolinensis)),(((((((((((((((Anolis_nebularia,(((((Aquila_cenchria,Lycaenopsis_garullus),Citharacanthus_novaeangliae),((Ethmostigmus_piscator,Sphenops_galactonotus),Heterodon_ulikovskii)),Madagascarophis_rusticolus),Pachydactylus_saxatilis)),(Chondropython_acuta,Uroplatus_valliceps)),Net_maldivarum),((Citharacanthus_flavomaculatus,Ctenosaura_vitticeps),Leiopython_carnivorus)),Chen_boschas),Pachytriton_nivalis),Pelodytes_leucoryphus),Cuon_novaeguineae),Limnaeus_tridactylum),Tringa_prominanus),Cuculus_taxus),(Phrynohyas_riparia,Porphyrio_duplex)),Cygnus_insignis),((((((Aplopeltura_vastus,Saiga_gregaria),(Colaeus_penelope,Desnana_hyperboreus)),Trionyx_tuberculosus),(Castor_laevis,((Desnana_iguana,Mustela_clypeata),Pelusios_coturnix))),(Gambelia_sphenocercus,Zosterops_leucopsis)),((Carabus_jacksoni,Megophrys_pica),Monticola_mitratus))),Petrocincla_anachoreta)),Eremophila_acuta),Dendrobates_alba),Hydrochelidon_climacophora),Ingerophrynus_metallica),(Buthus_sphenocercus,(Gypaetus_caucasicus,Madagascarophis_smaragdina))),((((((Chlidonias_kazanakowi,Cuon_alpinus),Rhacophorus_altaicus),Ninox_tristis),(Marmota_amboinensis,Totanus_musculus)),Salmo_venulosa),Mabuya_bukhunensis)),Amphiuma_reticulatus),Bombus_brachydactyla),(Callipogon_wislizeni,Meles_decor))),Acanthoscurria_calamita,(Alauda_spinifera,Chlidonias_melonotis));
Limnaeus_tridactylum Hottentotta_aceras

((((Ahaetulla_colubrinus,Allactaga_tadorna),(Porzana_tarandus,Tetraogallus_tadorna)),(((Anthropoides_peregusna,(Chelydra_hypoleucos,Leiopython_flammea)),Apus_albirostris),Odobenus_celeber)),((((((((((((Alaus_helvetica,Rhodostethia_cambridgei),Aquila_proteus),((((Aythia_garmani,Grampus_euptilura),Rissa_anachoreta),Nemachilus_chukar),Columba_striatus)),((((Cyriopagopus_opimus,Monodon_zonata),Spalerosophis_sirtalis),Xenochrophis_stimsoni),Hydrochelidon_bairdii)),Furcifer_livia),(((Alpes_plumipes,((Chlamydosaurus_molurus,sibiricus_barbata),(Dahurinaia_brevipes,Dahurinaia_pulcher))),((Asthenodipsas_hyperboreus,(Brachyramphus_penelope,Eirenis_sudanensis)),Pedostibes_arvensis)),Capella_latiscutatus)),(Lanius_oenanthe,Lepidobatrachus_longipennis)),((((((Buthus_erythropus,Gonyosoma_gregarius),Latastia_altaica),(Phalaropus_fissipes,Phrynomerus_regius)),(Lamprolepis_chamaeleontinus,Procellaria_leucostomum)),Panthera_amboinensis),((Melanocoryhpa_turneri,(Opheodrys_hirundo,Riparia_salei)),Pseudemys_mnemosyne))),((((Alloporus_tadorna,Chen_korschun),(Chlamydosaurus_dives,(Diomedea_cambridgei,(Nerodia_lineatus,Rhynchaspis_guttata)))),(Leiocephalus_anser,Saxicola_politus)),(((Ardea_mandarina,Totanus_hilarii),((Brachyramphus_calyptratus,Emberiza_major),Phyllopneuste_guineti)),Ardea_rusticolus))),(Cyclagras_platycephala,Thymallus_griseus)),((((Capeila_solitaria,Grammostola_geniculata),Pachydactylus_auratus),Phoca_leptochelis),((Dahurinaia_linaria,(Kinixys_latiscutatus,(((Phylloscopus_triangulum,Phylloscopus_vipio),Sterna_emilia),Rosalia_glareola))),Nemorhaedus_bedriagai))),Pyxicephalus_nyroca),((((Alaus_pelagicus,(((Ameiva_spaldingi,Pterocles_interpres),Vormela_serpentina),Telescopus_auratus)),Ursus_caeruleus),Thamnophis_rutila),(Alloporus_subrubrum,Fregilegus_castaneus)));
Buthus_erythropus Asthenodipsas_hyperboreus

(Acanthis_arenarius,(((((Acanthoceros_pholeter,Buthacus_ferrumequinum),(((Circaetus_melonotis,(Odobenus_boa,Rhynchaspis_aeruginosus)),Gerrhosaurus_papuana),(Hydrosaurus_laevis,Paramesotriton_pardus))),(((((((((Aphonopelma_morinellus,Saxicola_kingii),((Chrysopelea_virgo,Pseudorca_sujfunensis),Falco_tentaculatum)),Numenius_dahurica),Trapelus_agama),((((Aphonopelma_musicus,(Dyscophus_corone,Morelia_schrencki)),Cyclagras_lividum),((((((((((((Arctomys_taxus,Totanus_pardalis),Testudo_davidiana),Certhia_multifasciata),((Net_morinellus,Sitta_erythrogastra),Polypedates_penelope)),Vanellus_squamatus),((((Argynnis_macqueni,Balaenoptera_temminskii),Buthacus_paradisi),Gonocephalus_brandtii),((Aythya_taczanowskius,(Calidris_galericulata,Grus_dominus)),Latastia_getula))),Fuligula_gigas),Aythya_manul),Uromastyx_indica),Xenophrys_tolai),(((Cuora_parreyssi,Rhombomys_ornata),Rosalia_collaris),(Kassina_aristotelis,Opheodrys_xanthocheilus))),Citharacanthus_cocincinus)),Grammostola_percnopterus)),Hydrochelidon_longipes),Pogona_squaterola),Scorpio_ridibundus),Minipterus_multituberculatus)),((((((((((Acanthosaura_brachydactyla,((Latastia_maurus,Lobipes_geniculata),Vormela_rostratus)),(((Burhinus_graculus,Certhia_caniceps),Pica_scincus),Carabus_tigrinus)),Ninox_ferox),Neolycaena_politus),Caiman_varius),Hottentotta_celer),Tylototriton_sauritus),((Agama_hispida,Aythia_boyciana),Varanus_aestivus)),(((((((((Anthropoides_marinus,Chelodina_penelope),Podoces_euptilura),(Bronchocela_dactylisonans,(((Hemitheconyx_canorus,Porphyrio_tentaculatum),Ptychozoon_monorhis),Plegadis_carnifex))),Lamprophis_sphenocercus),Budytes_atthis),Phormictopus_dendrophila),Egretta_corsac),Laudakia_leschenaultii),((Furcifer_proteus,Tryngites_quinquetaeniata),Larus_cherrug))),Buthus_fasciolata)),Chamaeleo_dennysii),(Acanthogonatus_crucigera,Melanocorypha_heterolepidotus));
Chamaeleo_dennysii Podoces_euptilura

(((((Acanthogonatus_spaldingi,((((((((((Androctonus_drapiezii,Leptobrachium_dauricus),Chlamydosaurus_insularis),(Homopholis_reinwardti,Lobipes_rufinus)),crecca_pendulinus),(((Balaena_imperator,Net_rubida),Enhydra_cavimanus),Phelsuma_classicus)),Synthliboramphus_cristatellus),Eutamias_calyptratus),Uromastyx_flavomaculatus),Xenochrophis_corticale),((((Bombina_erythronota,Chrysemys_reticulatus),(Scaphiophryne_smithi,Turdus_turneri)),Pandinus_leucogeranus),Phylloscopus_piscator))),(Minipterus_alpestris,Terpsihone_aceras)),Casarca_indicus),Tringa_scincoides),((((((((((Acanthosaura_pygmeus,(((Casarca_dubia,Rhynchophis_himalayensis),Rhacophorus_boulengeri),(Onychodactylus_maurus,Scolopax_quadriocellata))),(Nemachilus_armata,Porzana_turneri)),Anser_caerulea),(((Aphonopelma_avinivi,(((Certhia_margaritifera,(Haliaetus_saiga,Marmota_taeniura)),Chamaeleo_clericalis),Ketupa_calvus)),Pelusios_carolinensis),Pelecanus_caudolineatus)),Pandion_lutra),Onychodactylus_czerskii),((((((((Allobates_lepturus,Podoces_subcinctus),Ameiva_margaritifera),(Mogera_musculus,Panthera_albatrus)),(((((((Eremophila_macrops,Totanus_vertebralis),Phrynomerus_oedicnemus),Pleurodeles_cancerides),(Gongylophis_equestris,Pterocles_amboinensis)),Haliaeetus_bicinctores),Gavia_caudatus),Scorpio_acanthinura)),Gypaetus_chrysaetus),Sphenops_margaritifera),Heteroscodra_exanthematicus),(Cygnopsis_peregrinus,Parnassius_calyptratus))),Capreolus_lutra),((((Allobates_bairdi,(Canis_melanuroides,((Homopholis_monacha,Kinixys_indica),Syrrhaptes_fulva))),((Calidris_chrysargos,(Gonocephalus_rosmarus,Moschus_monedula)),Chondropython_pusilla)),Capra_rutilans),((((Bombycilla_nipalensis,((((((Buthacus_erythropus,(Pagophila_scrofa,((Philacte_nipalensis,Porphyrio_mnemosyne),Pogona_macqueni))),(Philomachus_hungaricus,Thecla_glutinosus)),(Capeila_notaeus,(Kinosternon_bifasciatus,Parnassius_tetrix))),Cygnus_pica),((Equus_marmoratus,Phasianus_bedriagai),Squaterola_seemani)),Castor_chukar)),Citharacanthus_chuatsi),((Lycodon_dennysii,(Pyrgilauda_tentaculatum,Tadorna_docilis)),Ptyodactylus_querquedula)),Nyroca_gemmicincta))),(Megophrys_sieboldii,Tupinambus_aleutica)),Ahaetulla_salei);
Turdus_turneri Gonocephalus_rosmarus

((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((Trapelus_platyrhinos)Thymallus_caudatus)Tetraogallus_tetrax)Testudo_oedicnemus)Spizaetus_erythronotus)Siniperca_mutabilis)Scolopendra_paradisi)Scincus_altaica)Rhodostethia_classicus)Rhesus_gigas)Ptyodactylus_wislizeni)Porphyrio_weliczkowskii)Plethodon_carinatus)Platemys_porphyrio)Philothamnus_parahybana)Philacte_novaeguineae)Pelomedusa_tigrinus)Pelodytes_guttifer)Pelodiscus_maculatum)Pelecanus_saiga)Panthera_grossmani)Oxyura_breitensteini)Onychodactylus_salei)Oceanodroma_tanganicus)Nyroca_blakistoni)Numenius_dione)Notophthalmus_salamandra)Norops_albocinctus)Myotis_arvensis)Mogera_nasicus)Melanocoryhpa_azureus)Marmota_calyptratus)Madagascarophis_mehelyi)Lyrurus_grandis)Lycodon_mandarina)Liasis_aleutica)Lepus_caesius)Laudakia_colubrinus)Lagenorhynchus_krueperi)Kassina_giganteus)Ingerophrynus_dennysii)Hydrochelidon_geyri)Holaspis_nivalis)Hirundo_merganser)Hirundo_barroni)Halichoerus_boulengeri)Haliaeetus_geyri)Grammostola_oenanthe)Gonocephalus_angustirostris)Gongylophis_rufina)Gecarcinus_meermani)Ethmostigmus_odoratus)Erpeton_varius)Dipus_cianeus)Dendrobates_castaneus)Cypselus_stellatum)Cypselus_celer)Colaeus_perrotetii)Colaeus_glutinosus)Circus_schneideri)Chrttusia_haliaetus)Chondropython_sebae)Chalcides_hendricksoni)Castor_dentata)Carabus_dentata)Camptoloma_ferina)Buthacus_turtor)Burhinus_ferox)Burhinus_castaneus)Branta_dispar)Bombina_fernandi)Arenaria_guineti)Archispirostreptus_leucotus)Aquila_avinivi)Apalone_corone)Antaresia_leporosum)Anodonta_chamaeleontinus)Aix_peregrinus)Agama_plumipes)Aegialifes_sujfunensis)Aegialifes_chrysaetos)Acipenser_spaldingi)Acipenser_platyrhinos)Acanthosaura_paradisi);
Colaeus_glutinosus Agama_plumipes

((((((Acanthoceros_irregularis,Margaritifera_argentatus),((Bombina_albertisii,Paramesotriton_penelope),((((Cervus_cioides,Neolycaena_clypeata),Ursus_plumipes),Pachytriton_tinctorius),(Lutra_euptilura,((Ophisops_ruficollis,Vulpes_maurus),Phoca_krueperi))))),Homalopsis_guineti),Epipedobates_odoratus),Chlidonias_gregarius),((((((((((((((((((Aegialites_leucoptera,(((((Capeila_flavigularis,(Eremophila_tetrax,Onychodactylus_equestris)),(Picus_mnemosyne,Rhodostethia_iankowskii)),Leptobrachium_leucophyllata),Gavia_saiga),Ethmostigmus_catenifer)),Pagophila_fallax),Ptyodactylus_aestivus),Lampropeltis_collaris),((Buthus_nivicola,Ephibolus_picta),Lasiodora_hirundo)),Chlamydosaurus_bellii),Hemitheconyx_atthis),(Chalcides_laevis,((Eirenis_himalayensis,(Gambelia_albicilla,Rhabdophis_semipalmatus)),Gypaetus_marcianus))),Homalopsis_sanguinolentus),(Chlamydotis_dauricus,Telescopus_pelagicus)),Hirundo_sibilans),Limnaeus_ferina),Rangifer_scrofa),(((((((((((((Aquila_grunniens,Teratolepis_insignis),(Heterodon_cioides,Macrorhamphus_rutilans)),(((Canis_pallasii,Phrynohyas_erythronota),Net_vulpes),Philothamnus_lehmanni)),(Burhinus_laevis,Dipsosaurus_decorus)),((Eirenis_jacksoni,Kassina_galeatus),(Grampus_taxispilota,Pogona_schreibersi))),Monticola_physalus),Bos_felderi),Coregonus_ion),Saxicola_sibilans),Chondropython_melanoleucus),Sus_reticulatus),(Net_leiosoma,Phalaropus_circia)),((Clemmys_quinquetaeniata,Ctenosaura_mackloti),Ptychozoon_boschas))),((Chelydra_subrubrum,Trachemys_leuconotus),Equus_calamita)),Mustela_rutila),(((Himantopus_subrufa,Hottentotta_aleutica),Nemachilus_moschiferus),Varanus_pulcher)),(Asthenodipsas_minutus,Scorpio_battersbyi)),(((Amphiuma_leucopsis,Babycurus_tinnunculus),Limnaeus_mlokosiewiczi),Morelia_helvetica));
Pachytriton_tinctorius Picus_mnemosyne

(((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((Xenophrys_calligaster)Vulpanser_cristatus)Vipera_apollo)Uromastyx_pendulinus)Tropidurus_leucotus)Trapelus_caucasicus)Teratolepis_cynodon)Streptopelia_cyanogaster)Squaterola_bicinctores)Sphenurus_chrysargos)Spalax_querquedula)Spalax_elegans)Sitta_livia)Scincus_epops)Scaphiopus_cianeus)Salamandra_kurilensis)Saga_mutabilis)Rufibrenta_subminiatus)Riparia_turtur)Rhynchophis_arizonensis)Regulus_dactylisonans)Python_peregusna)Ptyodactylus_thibetanus)Ptychozoon_maldivarum)Ptychozoon_leucostomum)Pica_carolinensis)Phylloscopus_tarda)Phylloscopus_enydris)Phrynosoma_heterolepidotus)Phrynomerus_major)Phrynomerus_govinda)Phalacrocorax_geniculata)Pelecanus_cyanus)Passer_davidiana)Pagophila_lobatus)Pachytriton_cinaedus)Oenanthe_merganser)Odonthurus_calyptratus)Nipponia_indica)Ninox_borealis)Nemachilus_americanus)Morelia_wislizeni)Morelia_belliana)Mochlus_prasina)Micropalama_wogura)Megophrys_meermani)Margaritifera_labiatus)Mareca_gordoni)Macrorhamphus_jacksoni)Lystrophis_maritimus)Limosa_albicilla)Liasis_hendricksoni)Lepidobatrachus_ciliatus)Latastia_pendulinus)Kassina_monacha)Iguana_jubata)Hydrosaurus_fasciolata)Heterodon_tadorna)Haplopelma_undulata)Hadrurus_mehelyi)Hadogenes_nupta)Gypaetus_australis)Grus_ibera)Glareola_nigrolineatus)Gambelia_boa)Euspiza_ibis)Euspiza_constricticollis)Erpeton_tadorna)Erpeton_opimus)Epipedobates_clinatus)Dipsosaurus_perrotetii)Dipsosaurus_lesueurii)Dendrobates_jaculus)Dendrelaphis_decor)Dendrelaphis_calvus)Cypselus_sebae)Cygnus_caudolineatus)Crotaphytus_rutilans)Corvus_kingii)Chondropython_pardalis)Castor_macqueni)Canis_dentatus)Bronchocela_oenanthe)Bronchocela_imperator)Bombina_pallidus)Balaena_gregarius)Archispirostreptus_opimus)Alloporus_hendricksoni)Allobates_mutabilis)Agama_hungaricus)Aegypius_versicolor)Aegypius_paradisi)Aegypius_melanostictus);
Corvus_kingii Salamandra_kurilensis

(Acanthosaura_cliffordii,(((((((((((((((((((((Allactaga_marmoratus,Eudramias_flammea),Pelodytes_gibbosus),Panthera_longipennis),(((Arenaria_hendersoni,Egretta_leucoptera),Chrysemys_kuhli),(Leptopelis_marmoratus,Parus_ferina))),(Apalone_atthis,Colaeus_caniceps)),(Carabus_czerskii,Megaloperdix_cinereus)),Crotaphytus_grandis),Micropalama_kazanakowi),Pelodytes_stellatum),Corytophanes_hypomelus),((((Androctonus_cranwelli,Eumeces_scripta),((((Cypselus_tanganicus,Egretta_tinnunculus),Minipterus_elaphus),Dasypeltis_daurica),Pachytriton_pulcher)),((((Argynnis_tetrix,(Babycurus_lopatini,Eirenis_celer)),((Grampus_fiber,Tryngites_geniculata),Mochlus_tarandus)),(Candoia_prominanus,(Salvelinus_glottis,(Sturnus_lobatus,(Syrrhaptes_belliana,Vulpanser_anser))))),((Eremophila_obsoleta,Pseudorca_vulpes),Pterocles_clinatus))),((Elseya_dominicus,Mogera_subrufa),(Pica_maritimus,Sus_nippon)))),(((Anolis_maldivarum,((Atrophaneura_asperum,(((Cardiocranius_brachydactyla,Ctenotus_bellii),Cygnus_sphenocercus),(Dryobates_fluviatilis,Picus_calyptratus))),Petrocincla_aureola)),(((Candoia_alpinus,Ceratophrys_viscivorus),(((Cinclus_leucophyllata,Enhudra_gallicus),Geochelone_colchicus),Heterodon_horridum)),Gavia_daurica)),(Leiopython_albirostris,Ovis_pardus))),Corvus_diadema),Lamprophis_platyrhinos),(Ethmostigmus_lesueurii,Vormela_himantopus)),(Castor_scripta,((Falco_hasselquistii,Gerrhosaurus_parahybana),Phrynosoma_zonata))),(Pelomedusa_geyri,Sphenops_variegatus)),(Mogera_edulis,Saga_guttifer)),Chrysemys_cepediana),Chen_ruthveni),(Corallus_collectivus,((Felis_boa,Rangifer_chinensis),Pandinus_bonasus))),((Anas_coturnix,Coleonyx_fusca),((Ophisops_quinquestriatus,Petrocincla_flavigularis),Uromastyx_citreola)));
Corvus_diadema Megaloperdix_cinereus

((((((((((Abantias_glutinosus,Ephibolus_macqueni),(Arenaria_peregusna,Homopholis_sieboldii)),(Melanocoryhpa_perrotetii,Salmo_variegatus)),Boa_leucopsis),Bombus_aestivus),(Lepus_dives,(Prunella_galeatus,Siniperca_castaneus))),((Chettussia_fuliginosus,Ctenosaura_calamita),(Laudakia_felderi,Phormictopus_melanoleucus))),(((((Aplopeltura_zagrosensis,Sitta_ameiva),Philacte_mackloti),Porzana_galericulata),Litoria_caninus),Platalea_cranwelli)),(Ambystoma_epops,Buteo_reinwardti)),(((((((((((Aegialites_personata,(Grus_longicaudata,(Sitta_parreyssi,Vanellus_barbatus))),(Cyclemys_nigriceps,Tetrao_politus)),Philothamnus_rupestris),Cuculus_timidus),Terpsihone_doriae),Hysterocrates_turneri),(((((Arenaria_rutila,Eremophila_Jankowskii),(Boa_leucomelas,(Brachyramphus_similis,(Eudramias_monachus,((Geochelone_dendrophila,Rhesus_calidris),Phelsuma_collectivus))))),((Osmoderma_dolosus,Phormictopus_macqueni),Rhamphiophis_oxycephalum)),Lyrurus_mnemosyne),Holodactylus_siebenrocki)),(((((Antilope_cyanogaster,(Apus_baeri,Homopholis_opimus)),Salmo_erythropus),Lamprolepis_zonata),Sus_soloensis),Furcifer_flavolineata)),(((Chalcides_longicollis,Neolycaena_clarus),Phrynomerus_personata),Cynops_himalayanus)),Recurvirostra_salamandra),((((Anas_dexter,Pagophila_personata),((Lasiodora_parvus,Trionyx_dennysii),Scolopendra_fuscus)),(Aythya_pallidus,(Columba_doriae,((Emberiza_truncatus,Holodactylus_mongolica),Opheodrys_avocetta)))),((((Ctenotus_leuconotus,Eurynorhynchus_lobatus),Morelia_multifasciata),Nyctixalus_subrufa),Mylopharyngodon_compactus))),((Alopex_reticulatus,(Casarca_paradisi,Ketupa_tatarica)),((Capeila_botnica,(Pandion_doriae,Pyxicephalus_interiorata)),((Glareola_erythrogastra,Salamandra_mykiss),Leiolepis_mugodjaricus))));
Leiolepis_mugodjaricus Boa_leucopsis

(((((((((Acanthoscurria_albigula,((Chondropython_capra,Gyps_wogura),Ortigometra_vipio)),(((((((Bradypodion_tinnunculus,Leptopelis_aureostriata),(Mesoplodon_platyrhinos,(Oceanodroma_ferina,Uroplatus_meles))),Gambelia_buccata),(Python_anatina,Ruticilla_longicaudata)),Limosa_rusticolus),Ingerophrynus_arvensis),((Epicrates_garullus,Eudramias_decorus),Ursus_melanuroides))),((Ardea_vulpes,Homopholis_musculus),Megaptera_turtor)),Lepidobatrachus_shadini),Eunectes_krueperi),Parabuthus_cancerides),((((((((Anolis_mongolica,Osmoderma_intermedia),Gyps_diffidens),Chrysemys_schneideri),((Ardea_prasina,((((Brachypelma_dives,Corytophanes_jacksoni),Theloderma_regius),Telescopus_exanthematicus),((((Larus_franckii,Ursus_castus),Ophisops_cocincinus),Rangifer_dione),Nyroca_czerskii))),((Asthenodipsas_cyanogaster,Sphenurus_hilarii),(Phasianus_kuhli,Squaterola_pedo)))),(Fulica_vittatus,(Kinixys_oedicnemus,Saxicola_hypoleucos))),Uroplatus_albopillosum),Cervus_turtur),(Furcifer_atriceps,Nyroca_torquata))),(((((((((((((((((Antaresia_kuhli,Pelodiscus_hypoleucos),Moschus_reinwardti),Saxicola_flammea),Aplopeltura_galeatus),Gypaetus_physalus),Platemys_insignis),Chrysemys_piceus),Underwoodisaurus_manul),((Felis_himalayanus,Phrynosoma_horridum),(Laudakia_guttifer,Querquedula_relictus))),((((Aplopeltura_subrubrum,Bronchocela_cinaedus),((Prunella_jaspidea,Regulus_tetrix),Scaphiopus_tenuirostris)),Pusa_torquata),Eremophila_macqueni)),((Hadrurus_capreolus,Hadrurus_vegans),Micropalama_albertisii)),((Boiga_sinensis,Uroplatus_venulosa),Sus_uluguruensis)),Sturnus_caudolineatus),Pagophila_ignicapillus),Milvus_gregarius),Limnodromus_kazanakowi),Dendrobates_pulchra)),(Actitis_truncatus,(Upupa_angustirostris,Ziphius_deremensis)),(((((Allobates_subrufa,(Philothamnus_sanguinolentus,Pusa_hyperboreus)),(Pandinus_chrysaetus,Rufibrenta_duplus)),Physignathus_coelestinus),Recurvirostra_fischeri),(Neolycaena_arcticus,Psalmopoeus_brongersmai)));
Milvus_gregarius Antaresia_kuhli

((((((((((((((,((((,),),),)),),),),((,),)),),),((((((((((((,),),((,(,)),)),(,)),),),),(,)),),(,)),((,),)),((,(((((,),(,(,))),(,)),),)),(,)))),),),),),(((((,(((((((((((,(((((,),),),),)),),),),),),),),Hysterocrates_carbonaria),),)),(,)),(((((,),),),),(,))),),(((,(,)),),))),,(Alcedo_adspersus,));
Alcedo_adspersus Hysterocrates_carbonaria

((((Acanthosaura_aestivus,(((((Capra_pelagicus,Leiocephalus_ulikovskii),Ethmostigmus_tuberculosus),Chlamydosaurus_castaneus),Colaeus_cinereus),((Eudrornias_grus,Vormela_caniceps),Thecla_calvus))),((((Actitis_kopsteini,((((((Alloporus_schneideri,((((Aphonopelma_taxispilota,Chelus_aceras),Meles_rudicolis),Caiman_melanuroides),(Bombyx_alpinus,Monachus_glottis))),Tadorna_drapiezii),Spermophilus_schokari),Boiga_prominanus),Asthenodipsas_pallasii),((Atrophaneura_rufodorsata,Ovis_mutabilis),(Nemachilus_clypeatus,Ovis_clypeata)))),(Anthropoides_rhymnus,(Argynnis_colchicus,Fulica_dominicus))),Citellus_oedicnemus),(((Anser_metallica,Branta_sanguinolentus),Basiliscus_canus),(Gekko_wogura,Hemitheconyx_pusilla)))),Ethmostigmus_metallica),(((((((Acheron_sauromates,(Rhacodactylus_solitaria,Testudo_viscivorus)),Milvus_heliaca),Gongylophis_sepsoides),(Pyrrhocorax_keyzerlingii,Rhacophorus_salei)),Dafila_oedicnemus),Parnassius_ampullatus),(((Aquila_acanthinura,((Arenaria_sauromates,Gongylophis_marinus),((((((((((Aythya_gemmicincta,Hadrurus_stellatum),Rhesus_trianguligerus),Scorpio_calidris),(Phrynocephalus_caninus,(Phylloscopus_femoralis,(Physignathus_noctua,(Polypedates_leucopsis,Saga_longipennis))))),Spalerosophis_porphyrio),(Branta_truncatus,Siniperca_gecko)),Cygnus_cyanogenys),Capra_mlokosiewiczi),Pterinochilus_lopatini),Podoces_oenanthe))),Aquila_amethistina),((Coregonus_constrictor,Sturnus_salvator),Pyrrhocorax_dexter))),(((((((((Actitis_fernandi,(Chlidonias_meles,Phormictopus_spilota)),((((((Bombina_brevirostris,((Chondropython_subrufa,((Holodactylus_glutinosus,(Neolycaena_compactus,Scincus_caesius)),Natriciteres_deminutus)),Rangifer_trianguligerus)),Marmota_vittatus),Rhacodactylus_nelsonii),Parnassius_stagnalis),Dasypeltis_fissipes),(Circaetus_subcinctus,Onychodactylus_cingulata))),Butastur_pendulinus),Siniperca_doriae),Pelodytes_pardalis),((Aquila_obsoleta,Capra_maurus),Vormela_dendrophila)),Aegypius_graculus),Notophthalmus_salamandra),Anodonta_subcinctus));
Saga_longipennis Vormela_caniceps

(((((((((((((Acanthogonatus_arseniavi,((((((((((((((Anodonta_azureus,((Pareas_crispus,Rhynchaspis_spilota),Triturus_pulchra)),Aquila_carbonaria),Hydrosaurus_variegatus),Tamias_leucomelas),(Chlidonias_jaspidea,Ingerophrynus_cavimanus)),Thecla_lepturus),Neolycaena_vereda),(((Asthenodipsas_stimsoni,Hirundo_leptochelis),Hadogenes_longicaudata),Leiopython_spaldingi)),(Bradypterus_bengalensis,Chrysopelea_mackloti)),Upupa_stagnalis),Leptobrachium_nupta),((((Callipogon_medici,Hyperoodon_duplus),((Cypselus_collectivus,Leiocephalus_oxycephalum),((Hysterocrates_fasciata,((Netta_altaica,Pachydactylus_hosii),(Nyroca_fulva,(Recurvirostra_variegatus,Sericinus_cingulata)))),Mareca_pardus))),(Candoia_cristatus,((((((Candoia_oenanthe,Chettussia_hilarii),Tiliqua_vitticeps),(Onychodactylus_uluguruensis,Sphenops_oxycephalum)),Marmota_rufinus),Chelydra_chamaeleontinus),(Nemorhaedus_melleri,Phormictopus_purpurascens)))),Geochelone_nasuta)),((((Bronchocela_erythropus,Lampropeltis_aceras),Rhodostethia_barbata),Morelia_enhydris),Rufibrenta_cavimanus)),Lutra_pictus)),(Haliaeetus_capra,Querquedula_zagrosensis)),(Hadogenes_ladogensis,Pogona_aeruginosus)),(((Asthenodipsas_chrysaetus,((Epipedobates_tenuirostris,((((((Leptopelis_monedula,(Otis_colubrinus,(Passer_dactylisonans,Phormictopus_dulkeitiana))),Oligodon_monilis),Phormictopus_cyanus),Terpsihone_ferina),Scaphiopus_durus),Odonthurus_plumipes)),((Eutamias_onocrotalus,Sitta_cliffordii),Middendorffinaia_montela))),Riparia_schreibersi),Otocoris_salvator)),Neolycaena_sinensis),(Chrysopelea_baibacina,(Coregonus_madagascariensis,Phasianus_cygnoides))),Podoces_temminckii),Phalacrocorax_sibilans),Leiurus_dennysii),(Ceratophrys_situla,(Osteopilus_monacha,Varanus_purpurascens))),Phrynohyas_strepera),(((Aegypius_schneideri,(((Boiga_gratiosa,Otis_rudicolis),((Caiman_nigra,Podoces_dulkeitiana),((Rhinolophus_reinwardti,Strepsilas_marinus),Tylototriton_fiber))),((Damon_ferox,Dryobates_uncia),Riparia_canorus))),Apus_corsac),Hyla_mexicanum)),Acipenser_tarda,((Aegialites_tigris,Homopholis_guttifer),Osteopilus_bedriagai));
Varanus_purpurascens Bronchocela_erythropus

(((((((((((,),((,(((Ciconia_pictus,),(,((,),Scaphiopus_temminckii))),)),)),),((((,),),),)),),(,)),),),(((,((,),)),),)),((((((,((((,),(,)),((,),)),)),),),),),)),(((((((,(,)),),((((((,),),),(((((,),),),),(,))),),)),(,)),),((,),)),((((((,(,(,(,(,))))),),(((,((,),)),),)),(,)),),)),);
Ciconia_pictus Scaphiopus_temminckii

"""

In [ ]:
print(get_answer(input_txt))